# Interactive machine-learning analysis

The automated pipeline stops after it selects consensus DMS and writes feature matrices for the fixed training, threshold-calibration, and deployment cohorts. Run these notebook cells when you are ready to explore model fitting. The deployment cohort is held out for final evaluation: do not use its results to choose features, tune the model, or adjust the threshold.

The expected cohorts are 19 training mothers, 6 calibration mothers (3 infected and 3 uninfected), and 4 deployment mothers. All four hatchlings from each mother stay in the same cohort.

## Load the generated data

Set `CH2_WORKDIR` in the notebook environment to the pipeline data directory before running this cell. The matrices are CSV files with DMS as rows and hatchlings as columns.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

workdir = Path(os.environ["CH2_WORKDIR"]).expanduser().resolve()
ml_dir = workdir / "output" / "MLdata"
results_dir = workdir / "output" / "MLresults"

def load_cohort(name):
    feature_path = ml_dir / f"hatch_{name}_DMS_percMeth_final.csv"
    metadata_path = ml_dir / f"hatch_{name}_metadata_final.csv"
    features = pd.read_csv(feature_path, index_col=0).T
    metadata = pd.read_csv(metadata_path, dtype={"Sample": str, "Mother": str})
    metadata = metadata.set_index("Sample")
    if not features.index.equals(metadata.index):
        raise ValueError(f"{name}: feature and metadata sample order differs")
    if not features.columns.is_unique or not features.index.is_unique:
        raise ValueError(f"{name}: duplicate DMS or sample IDs")
    X = features.apply(pd.to_numeric, errors="raise")
    y = pd.to_numeric(metadata["parasiteFactor"], errors="raise").astype(int)
    if set(y.unique()) != {0, 1}:
        raise ValueError(f"{name}: expected both response classes 0 and 1")
    return X, y, metadata

X_train, y_train, train_meta = load_cohort("modelTraining")
X_calibration, y_calibration, calibration_meta = load_cohort("thresholdCalibration")
X_deployment, y_deployment, deployment_meta = load_cohort("deployment")

if not X_train.columns.equals(X_calibration.columns) or not X_train.columns.equals(X_deployment.columns):
    raise ValueError("Cohorts do not contain the same consensus DMS in the same order")

cohort_mothers = [set(meta["Mother"]) for meta in (train_meta, calibration_meta, deployment_meta)]
if any(cohort_mothers[i] & cohort_mothers[j] for i in range(3) for j in range(i + 1, 3)):
    raise ValueError("A maternal group occurs in more than one cohort")

pd.DataFrame(
    {
        "samples": [len(y_train), len(y_calibration), len(y_deployment)],
        "mothers": [train_meta.Mother.nunique(), calibration_meta.Mother.nunique(), deployment_meta.Mother.nunique()],
        "uninfected_samples": [(y == 0).sum() for y in (y_train, y_calibration, y_deployment)],
        "infected_samples": [(y == 1).sum() for y in (y_train, y_calibration, y_deployment)],
    },
    index=["training", "threshold calibration", "deployment"],
).assign(consensus_DMS=X_train.shape[1])

## Tune and fit a model

This example uses elastic net and mother-grouped cross-validation for hyperparameter tuning. Edit the estimator or parameter grid here to explore other models. The CV score tunes the model after consensus DMS selection; it is not an unbiased estimate of the complete feature-selection workflow.

In [ ]:
pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                class_weight="balanced",
                max_iter=10000,
                random_state=42,
            ),
        ),
    ]
)

parameter_grid = {
    "model__C": np.logspace(-4, 4, 10),
    "model__l1_ratio": [0.1, 0.25, 0.5, 0.75, 0.9],
}
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    pipeline,
    parameter_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
)
search.fit(X_train, y_train, groups=train_meta["Mother"])

print("Best parameters:", search.best_params_)
print("Best grouped-CV ROC AUC:", search.best_score_)
model = search.best_estimator_

## Select a classification threshold

Use only the six calibration mothers to choose a threshold. This example maximizes balanced accuracy; inspect the predictions and change the criterion if your scientific objective differs. With few calibration mothers, the selected threshold may be uncertain.

In [ ]:
calibration_probabilities = model.predict_proba(X_calibration)[:, 1]
thresholds = np.unique(np.r_[0.0, calibration_probabilities, 0.5, 1.0])
threshold_scores = [
    balanced_accuracy_score(y_calibration, calibration_probabilities >= threshold)
    for threshold in thresholds
]
best_score = max(threshold_scores)
best_candidates = [
    threshold
    for threshold, score in zip(thresholds, threshold_scores)
    if np.isclose(score, best_score)
]
optimal_threshold = min(best_candidates, key=lambda value: (abs(value - 0.5), value))

calibration_predictions = pd.DataFrame(
    {
        "Mother": calibration_meta["Mother"],
        "observed": y_calibration,
        "probability": calibration_probabilities,
        "predicted": (calibration_probabilities >= optimal_threshold).astype(int),
    },
    index=calibration_meta.index,
)
print("Selected threshold:", optimal_threshold)
print("Calibration balanced accuracy:", best_score)
calibration_predictions

## Freeze the model, then evaluate deployment

Run this cell to save the fitted model and calibration threshold before inspecting deployment performance. The deployment cohort is evaluated once and must not be used to revise the model or threshold.

In [ ]:
import joblib

results_dir.mkdir(parents=True, exist_ok=True)
model_path = results_dir / "interactive_elastic_net_model.joblib"
joblib.dump(
    {
        "estimator": model,
        "threshold": float(optimal_threshold),
        "feature_ids": X_train.columns.tolist(),
        "best_params": search.best_params_,
    },
    model_path,
)
print("Saved frozen model:", model_path)

In [ ]:
deployment_probabilities = model.predict_proba(X_deployment)[:, 1]
deployment_predictions = (deployment_probabilities >= optimal_threshold).astype(int)
deployment_results = pd.DataFrame(
    {
        "Mother": deployment_meta["Mother"],
        "observed": y_deployment,
        "probability": deployment_probabilities,
        "predicted": deployment_predictions,
    },
    index=deployment_meta.index,
)
print("Deployment balanced accuracy:", balanced_accuracy_score(y_deployment, deployment_predictions))
print("Deployment accuracy:", accuracy_score(y_deployment, deployment_predictions))
print("Deployment ROC AUC:", roc_auc_score(y_deployment, deployment_probabilities))
print("Confusion matrix [rows=true, columns=predicted]:")
print(confusion_matrix(y_deployment, deployment_predictions, labels=[0, 1]))
deployment_results